# 07 — Serie temporali II: smoothing esponenziale, damped trend, Kalman (1950s–1985)
Kernel **JupyterNet** (C#) su runtime **.NET 11**. Prerequisito: `dotnet build src/MathemaSharp.slnx -c Release`. Da terminale: `scripts/run-notebooks.ps1` (imposta il roll-forward al runtime 11); in VS Code serve `DOTNET_ROLL_FORWARD=LatestMajor` e, finché .NET 11 è RC, `DOTNET_ROLL_FORWARD_TO_PRERELEASE=1`.

Prima si aggiorna il **livello**, poi si aggiungono **trend** (Holt, 1957) e **stagionalità** (Winters, 1960). Nel 1960 il **filtro di Kalman** formalizza il «prevedi e correggi» su uno stato nascosto. Le due famiglie risultano la stessa cosa vista da due lati.

In [ ]:
#load "../shared/MathemaSharp.csx"
using System.Numerics;
using MathemaSharp.Core;
using MathemaSharp.TimeSeries;
// Local level: livello che cammina a caso (varianza q) osservato con rumore (varianza r).
const double q = 0.5, r = 2.0;
var rng = new Random(1960);
var level = 0.0;
var truth = new double[200];
var obs = new double[200];
for (var t = 0; t < 200; t++)
{
    level += Math.Sqrt(q) * Simulation.StandardNormal(rng);
    truth[t] = level;
    obs[t] = level + Math.Sqrt(r) * Simulation.StandardNormal(rng);
}
Chart.Lines("Local level: livello nascosto e osservazioni", new Line("osservazioni", obs), new Line("livello vero", truth));

## SES: una media con pesi geometrici
$$\ell_t = \alpha y_t + (1-\alpha)\ell_{t-1} \;\Longrightarrow\; \ell_t = \sum_{j\ge0}\alpha(1-\alpha)^j y_{t-j}, \qquad \hat y_{T+h}=\ell_T.$$
$\alpha$ si stima minimizzando l'SSE degli errori a un passo. `FitSesAlpha` usa la **sezione aurea**: ogni iterazione riduce l'intervallo del fattore $1/\varphi\approx0.618$ riusando un punto già valutato. Presuppone un SSE unimodale in $\alpha$.

In [ ]:
var grid = Enumerable.Range(1, 99).Select(i => i / 100.0).ToArray();
Chart.Lines("SSE a un passo in funzione di α (asse x: α × 100)", new Line("SSE", grid.Select(a => ExponentialSmoothing.Ses(obs, a, 0).Sse).ToArray(), Start: 1));
var alphaHat = ExponentialSmoothing.FitSesAlpha(obs);
var kStar = KalmanFilter.LocalLevelSteadyStateGain(q, r);
Console.WriteLine($"α stimato (sezione aurea) = {alphaHat:F4}");
Console.WriteLine($"K* teorico (Riccati, q/r = {q / r}) = {kStar:F4}");

## Kalman: perché $\hat\alpha \approx K^*$
Il local level è $\ell_t=\ell_{t-1}+\eta_t$, $y_t=\ell_t+\varepsilon_t$. Il filtro alterna:
- **predict**: $P^- = P + q$;
- **update**: $K = P^-/(P^-+r)$, $\ell \leftarrow \ell + K(y-\ell)$, $P = (1-K)P^-$.

L'update del livello è *esattamente* una riga di SES con $\alpha=K$. A regime $P^-$ soddisfa la Riccati $P = P - P^2/(P+r) + q$, ovvero $P^2-qP-qr=0$:
$$P^* = \frac{q+\sqrt{q^2+4qr}}{2}, \qquad K^* = \frac{P^*}{P^*+r}.$$
Quindi **SES = filtro di Kalman stazionario del local level**, e $\alpha$ dipende solo dal rapporto segnale/rumore $q/r$. Stimare $\alpha$ con l'SSE equivale a stimare $q/r$. La libreria usa la covarianza in forma di **Joseph**, $(I-KH)P(I-KH)^\top + KRK^\top$: algebricamente uguale, ma resta simmetrica e semidefinita positiva anche con gli arrotondamenti.

In [ ]:
var kf = KalmanFilter.LocalLevel(q, r, obs[0], 1e6);
var filtered = kf.Filter(obs);
Chart.Lines("Guadagno K_t: dal prior diffuso (K≈1) al regime K*", new Line("K_t", filtered.FirstGain[..30]), new Line("K*", Enumerable.Repeat(kStar, 30).ToArray(), Dashed: true));
var ses = ExponentialSmoothing.Ses(obs, kStar, 0);
Console.WriteLine($"max |Kalman − SES(α=K*)| sulle previsioni a un passo, t ≥ 50: {Enumerable.Range(50, 150).Max(t => Math.Abs(filtered.Predictions[t] - ses.Fitted[t])):E2}");

## Dati mancanti: il vantaggio del modello di stato
Con un'osservazione mancante il Kalman salta l'update: lo stato resta la previsione e la varianza $S_t = P^-_t + r$ cresce di $q$ a ogni passo vuoto. Lo smoothing esponenziale "a ricetta" non ha un modo naturale per farlo. Il filtro restituisce anche la log-verosimiglianza per decomposizione dell'errore di previsione, $-\tfrac12\sum(\ln 2\pi S_t + v_t^2/S_t)$: è ciò che si massimizza per stimare $q$ e $r$.

In [ ]:
var gappy = (double[])obs.Clone();
for (var t = 80; t < 110; t++) gappy[t] = double.NaN;
var fg = KalmanFilter.LocalLevel(q, r, obs[0], 1e6).Filter(gappy);
// Da t = 5: con il prior diffuso (varianza 1e6) la banda iniziale è enorme e schiaccerebbe la scala.
var upper = fg.Predictions.Select((p, t) => p + 2 * Math.Sqrt(fg.PredictionVariances[t])).ToArray();
var lower = fg.Predictions.Select((p, t) => p - 2 * Math.Sqrt(fg.PredictionVariances[t])).ToArray();
Chart.Lines("Previsione a un passo ± 2√S con 30 osservazioni mancanti (t = 80…109)",
    new Line("osservazioni", gappy[5..], Start: 5),
    new Line("previsione", fg.Predictions[5..], Start: 5),
    new Line("+2√S", upper[5..], Start: 5, Dashed: true),
    new Line("−2√S", lower[5..], Start: 5, Dashed: true));
Console.WriteLine($"log-verosimiglianza: q=0.5 → {KalmanFilter.LocalLevel(0.5, 2, obs[0], 1e6).Filter(obs).LogLikelihood:F2};  q=0.05 → {KalmanFilter.LocalLevel(0.05, 2, obs[0], 1e6).Filter(obs).LogLikelihood:F2};  q=5 → {KalmanFilter.LocalLevel(5, 2, obs[0], 1e6).Filter(obs).LogLikelihood:F2}");

## Holt e il damped trend (Gardner & McKenzie, 1985)
$$\hat y_{t|t-1}=\ell_{t-1}+\phi b_{t-1},\quad \ell_t=\alpha y_t+(1-\alpha)(\ell_{t-1}+\phi b_{t-1}),\quad b_t=\beta(\ell_t-\ell_{t-1})+(1-\beta)\phi b_{t-1}$$
$$\hat y_{T+h}=\ell_T+(\phi+\phi^2+\dots+\phi^h)\,b_T \;\xrightarrow{h\to\infty}\; \ell_T+\frac{\phi}{1-\phi}b_T.$$
Con $\phi=1$ è Holt lineare: il trend prosegue all'infinito. Con $\phi<1$ il contributo del trend si riduce agli orizzonti lunghi. Empiricamente (M-competitions) è uno dei metodi più robusti: raramente il migliore, quasi mai disastroso.

In [ ]:
// Crescita che rallenta: la pendenza decade geometricamente.
var slowing = Enumerable.Range(0, 60).Select(t => 50 + 40 * (1 - Math.Pow(0.96, t)) + Simulation.StandardNormal(rng)).ToArray();
double[] tr = slowing[..40], te = slowing[40..];
var holt = ExponentialSmoothing.Holt(tr, 0.4, 0.2, 20);
var damped = ExponentialSmoothing.Holt(tr, 0.4, 0.2, 20, phi: 0.85);
Chart.Lines("Holt lineare vs damped (φ = 0.85)", new Line("osservato", slowing), new Line("Holt", holt.Forecast, Start: 40, Dashed: true), new Line("damped", damped.Forecast, Start: 40, Dashed: true));
Table.Show(new[] { "Metodo", "RMSE h=1..20", "limite h→∞" }, new[]
{
    new object[] { "Holt (φ=1)", Accuracy.Rmse(te, holt.Forecast), "∞" },
    new object[] { "Damped (φ=0.85)", Accuracy.Rmse(te, damped.Forecast), (damped.Level + 0.85 / 0.15 * damped.Trend).ToString("F2") },
});

## Holt-Winters additivo (1960)
Aggiunge $s_t=\gamma(y_t-\ell_{t-1}-\phi b_{t-1})+(1-\gamma)s_{t-m}$ e $\hat y_{T+h}=\ell_T+\phi_h b_T+s_{T+h-m(k+1)}$. Inizializzazione: la media del primo ciclo stima il livello al **centro** del ciclo, $(m-1)/2$, non alla fine. Spostandola lungo il trend, retta + stagionalità a media nulla viene riprodotta esattamente: è uno dei test della libreria.

In [ ]:
var seasonalY = Enumerable.Range(0, 96).Select(t => 20 + 0.3 * t + 5 * Math.Sin(2 * Math.PI * t / 12) + Simulation.StandardNormal(rng)).ToArray();
var hw = ExponentialSmoothing.HoltWinters(seasonalY[..84], 12, 0.3, 0.05, 0.2, 12);
Chart.Lines("Holt-Winters: fitted a un passo e previsione a 12 mesi", new Line("osservato", seasonalY), new Line("fitted", hw.Fitted), new Line("previsione", hw.Forecast, Start: 84, Dashed: true));
Console.WriteLine($"MASE: Holt-Winters {Accuracy.Mase(seasonalY[84..], hw.Forecast, seasonalY[..84], 12):F3}, seasonal naive {Accuracy.Mase(seasonalY[84..], ReferenceForecasts.SeasonalNaive(seasonalY[..84], 12, 12), seasonalY[..84], 12):F3}");

## ETS e ARIMA: famiglie complementari
SES equivale ad ARIMA(0,1,1) con $\theta=\alpha-1$: da $\ell_t = y_t-(1-\alpha)e_t$ segue $\hat y_{t+1}=y_t+(\alpha-1)e_t$. Holt lineare equivale ad ARIMA(0,2,2). Il quadro ETS in spazio degli stati (Hyndman et al., 2002) unifica smoothing e verosimiglianza. ETS e ARIMA non sono uno l'aggiornamento dell'altro: si sovrappongono solo sui modelli lineari.

In [ ]:
var sesF = ExponentialSmoothing.Ses(obs, 0.3, 5).Forecast;
var arimaF = new ArimaModel<double>(0, 1, 1, 0, Array.Empty<double>(), new[] { 0.3 - 1 }, 1).Forecast(obs, 5);
Console.WriteLine($"SES(α=0.3):            {string.Join(", ", sesF.Select(v => v.ToString("F10")))}");
Console.WriteLine($"ARIMA(0,1,1), θ=−0.7:  {string.Join(", ", arimaF.Select(v => v.ToString("F10")))}");